# Topic: Advanced RAG Architectures (HyDE, Parent-Child, Sentence-Window, Multi-Query)

## Definition (30-second explanation)
*   **Naive RAG** is asking a librarian a vague question and hoping their first keyword lookup finds the exact page you need.
*   **HyDE (Hypothetical Document Embeddings)** is hallucinating what the answer *probably* looks like first, then searching the library for real documents that match that hallucinated answer's structure and vocabulary.
*   **Parent-Child Chunking** splits text into tiny paragraphs for razor-sharp vector search, but swaps them for the entire chapter when feeding the LLM so it never loses context.
*   **Sentence-Window Retrieval** embeds single isolated sentences, but retrieves the surrounding $k$ sentences before and after it at inference time. 
*   **Multi-Query Expansion** asks the same question in 4 different ways so you don't miss documents due to poor user phrasing.

## Why Interviewers Ask This
*   Naive RAG (embed chunk $\rightarrow$ search $\rightarrow$ send to LLM) fails in production 30% to 50% of the time due to query-document semantic mismatch, lost context, or fragile phrasing.
*   Interviewers want to see if you can architect advanced retrieval strategies that decouple *what you search on* from *what you pass to the generator*.

## Core Concepts (The 3-Layer Anatomy)
*   **The Bottleneck:**
    *   *Asymmetric Query-Doc Problem:* A 6-word question ("Fix error code 404") has a completely different vector structure than a 500-word troubleshooting manual passage.
    *   *Search vs. Context Dilemma:* Small chunks embed accurately but lack context for generation; large chunks have great context but create blurry, diluted vector averages that ruin search accuracy.
*   **The Mechanism:**
    *   **HyDE:** Pass user query $Q \rightarrow$ lightweight LLM generates a fake answer $\hat{D} \rightarrow$ embed $\hat{D}$ (Document-to-Document search instead of Query-to-Document) $\rightarrow$ retrieve real documents.
    *   **Parent-Child:** Embed small child chunks (e.g., 200 tokens) in Vector DB $\rightarrow$ link metadata `parent_id` to large parent chunk (e.g., 2000 tokens) in a DocStore $\rightarrow$ on hit, fetch parent for LLM generation.
    *   **Sentence-Window:** Embed single sentences in Vector DB $\rightarrow$ at query time, look up window buffer ($[-k, +k]$ sentences) in document store $\rightarrow$ inject full paragraph window into prompt.
    *   **Multi-Query:** Pass query to LLM to generate 3–5 diverse variations $\rightarrow$ retrieve top candidates for each $\rightarrow$ deduplicate and combine via reciprocal rank fusion (RRF).
*   **The Trade-off:**
    *   HyDE introduces extra LLM inference latency and will retrieve completely wrong documents if the model hallucinates wildly on niche domain topics.
    *   Multi-Query multiplies Vector DB queries and latency by $N\times$.
    *   Parent-Child and Sentence-Window require managing a dual-storage infrastructure (Vector DB + Key-Value Document Store).

## When to Use
*   **HyDE:** Vague, open-ended conceptual queries or when user questions are short and structurally asymmetric to document passages.
*   **Parent-Child Chunking:** Structured documents (contracts, manuals, research papers) where fine-grained precision is needed to locate facts, but broad narrative context is needed to synthesize answers.
*   **Sentence-Window Retrieval:** Highly dense technical or legal texts where individual sentences contain critical definitions that must be read within their immediate paragraph context.
*   **Multi-Query Expansion:** User-facing consumer chatbots where users type ambiguous, conversational, or sloppy prompts.

## Advantages
*   Bridges the embedding vector space gap between short questions and long answer paragraphs (HyDE).
*   Completely eliminates the tension between chunk size for indexing vs. chunk size for generation (Parent-Child & Sentence-Window).
*   Drastically boosts Retrieval Recall@K without needing to re-train or fine-tune embedding models.
*   Resilient against poor user query formulation (Multi-Query).

## Limitations
*   Increases end-to-end latency: HyDE and Multi-Query require an extra LLM call *before* search even starts.
*   HyDE failure mode: If the LLM generates a plausible hallucination with fictitious entities, the vector search retrieves irrelevant real documents matching the fiction.
*   Storage complexity: Requires keeping a synchronized Key-Value store (Redis, Mongo, SQLite) alongside the vector database.

## Common Comparisons
*   **HyDE vs. Query Expansion:** Query expansion generates synonyms or alternative questions; HyDE generates a complete *hypothetical answer passage* to perform doc-to-doc vector search.
*   **Parent-Child vs. Sentence-Window:** Parent-Child uses hierarchical chunks (child paragraph $\rightarrow$ parent section/document); Sentence-Window strictly targets a sliding window of neighboring sentences around a single matching sentence.

## Common Interview Traps
*   **Feeding the HyDE Fake Document to the Reader LLM:** The hypothetical document generated by HyDE is *only* used for embedding and retrieval. It is thrown away immediately after search; only genuine retrieved documents are passed to the final LLM prompt.
*   **Using HyDE for Exact Fact Retrieval:** If a user asks "What was our Q3 2025 revenue for Project Titan?", the LLM cannot possibly guess the exact number. HyDE will hallucinate a fake number, search for documents with that fake number, and fail.

## Python Syntax (LangChain HyDE Implementation)
```python
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import HypotheticalDocumentEmbedder
from langchain_community.vectorstores import Chroma

# 1. Base LLM to generate hypothetical answer & base embedding model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
base_embeddings = OpenAIEmbeddings()

# 2. Wrap embeddings with HyDE generator
hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=base_embeddings,
    prompt_key="web_search"  # Built-in prompt template instructing LLM to write a mock passage
)

# 3. Vector store initialized with HyDE embedder
vectorstore = Chroma(collection_name="docs", embedding_function=hyde_embeddings)

# 4. Search automatically triggers: User Query -> Mock Doc -> Vector Search
retrieved_docs = vectorstore.similarity_search("How does speculative decoding work?")
```

## Important Formula
*   **Document-to-Document Similarity (HyDE):**
    $$\text{Score}(Q, D) = \text{Cosine}(\vec{E}(\text{LLM}(Q)), \vec{E}(D))$$
    *(Replaces asymmetric query-doc similarity $\text{Cosine}(\vec{E}(Q), \vec{E}(D))$ with symmetric doc-doc similarity).*

## 45-Second Interview Answer
"Naive RAG suffers from two fundamental flaws: asymmetric vector spaces between short queries and long passages, and the conflict between small chunk sizes for search versus large chunk sizes for generation. To resolve this, I implement advanced RAG patterns. For query mismatch, I use **HyDE** to generate a hypothetical answer passage first, transforming the search into a symmetric document-to-document match. For context preservation, I use **Parent-Child Chunking** or **Sentence-Window Retrieval**, where we index small child sentences into the vector database for pinpoint retrieval, but swap them for the larger parent document or surrounding sentence window before prompt assembly. This maximizes vector precision without sacrificing context."

## Practice Question:

### Q1: HyDE Hallucination Traps & Multi-Query Scaling
**Question:** An engineer proposes using HyDE and a 10-query Multi-Query Expansion on every request across two systems: System A (an internal financial auditing bot querying private Q3 metrics) and System B (a customer onboarding bot answering vague "how-to" questions). Why will HyDE fail on System A, and what are the production bottlenecks and fixes for a blanket 10-query expansion?

**Answer:**
1. **Why HyDE Fails on System A (Private/Exact Data) vs. System B:**
   * **System B (Success):** For vague conceptual queries (*"How do I get my team started?"*), the LLM has strong prior knowledge of software onboarding concepts. Its hypothetical answer shares the exact vocabulary and structure of real onboarding documentation, bridging the query-to-document vector gap.
   * **System A (Failure Mode):** Because the LLM was never trained on private internal Q3 2024 financial reports, it has zero knowledge of the actual figures or internal codenames. HyDE will hallucinate plausible-sounding but completely fabricated numbers and terms, shifting the embedding vector toward irrelevant documents. For System A, **Hybrid Search (Dense + BM25)** is far superior to lock onto exact metric names, dates, and business units.
2. **Multi-Query Production Bottlenecks & Optimization:**
   * **Latency & QPS Amplification:** A 10-query expansion forces a sequential LLM generation call *before* retrieval even starts (adding 500ms–1s latency), followed by $11\times$ the embedding API calls, $11\times$ the Vector DB QPS load, and a larger candidate pool to deduplicate and merge via RRF.
   * **Architectural Fix:**
     1. **Adaptive Query Routing:** Use a fast intent classifier or router step to evaluate query specificity—clear, specific queries bypass expansion entirely ($N=1$), while vague queries trigger a capped expansion ($N=3\text{ to }4$).
     2. **Async Parallel Execution:** Execute the expanded embedding and vector search calls concurrently using `asyncio` so wall-clock search latency does not scale linearly with $N$.

**Interview Tips:**
*   **Rule of Thumb for HyDE:** HyDE only works when the base LLM *already knows the general domain* well enough to fake a realistic answer structure. Never use it for private acronyms, proprietary codenames, or exact numbers.
*   **Async vs. QPS:** Always distinguish between *wall-clock latency* (solved by async parallel search) and *database QPS load* (still multiplied by $N$ searches).

### Q2: Parent-Child vs. Sentence-Window & Boundary Edge Cases
**Question:** Compare the storage and context-expansion mechanics of Parent-Child Chunking vs. Sentence-Window Retrieval. What happens in Parent-Child if 4 matched children belong to the same parent, and what happens if a critical rule sits at the bottom boundary of Parent #1 while its exception sits at the top of Parent #2?

**Answer:**
1. **Storage & Expansion Mechanics:**
   * **Parent-Child Chunking:** Splits documents into static hierarchical blocks (e.g., 2,000-token Parents and 200-token Children). Children are embedded in the Vector DB with a `parent_id` metadata pointer, while Parents live in a separate Key-Value Document Store. On retrieval, matched children are swapped for their parent document via a secondary ID lookup.
   * **Sentence-Window Retrieval:** Splits text into single isolated sentences for razor-sharp embedding. During ingestion, the surrounding $\pm k$ sentences (e.g., 3 sentences before and 3 after) are saved directly inside that vector's **metadata payload** (`window`). At query time, no secondary database lookup is required—a post-processor simply replaces the matched sentence with its `metadata["window"]` string before calling the LLM.
2. **Edge-Case Handling:**
   * **Multiple Children / Same Parent:** The `ParentDocumentRetriever` automatically **deduplicates** by `parent_id`. If 4 out of 5 top child chunks belong to `Parent_#1`, `Parent_#1` is fetched and injected into the LLM prompt only once.
   * **The Boundary Cutoff Problem:** Because Parent-Child uses fixed static boundaries, a matched child at the very end of `Parent_#1` will fail to capture an exception sitting at the start of `Parent_#2` (unless heavy parent overlap is configured). **Sentence-Window** solves this naturally because the window is always *centered dynamically on the matched sentence*, grabbing the $k$ preceding and $k$ succeeding sentences regardless of where artificial chunk boundaries would have fallen.

**Interview Tips:**
*   **Infra Comparison:** Point out that Sentence-Window only needs **1 database** (Vector DB + metadata payload, at the cost of duplicated text in metadata), whereas Parent-Child needs **2 databases** (Vector DB + Key-Value DocStore, with zero text duplication).
*   **Dynamic Centering:** Contrast Parent-Child's *static boundaries* with Sentence-Window's *sentence-centered sliding window*.